# Multiple Comparisons and the Bonferroni Correction

**DS4DH Practice Pack · Module 04 — Statistical Inference**

*Technique:* Family-wise error rate and the Bonferroni correction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/04b_bonferroni.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

A p-value of 0.05 means: if nothing is going on, this result or stronger turns up
5% of the time.

Run one test and that is a 1-in-20 risk. Run four and, if nothing is going on
anywhere, the chance that *at least one* crosses 0.05 is much higher than 1 in 20.
Reporting that one as a finding is the most common statistical error in applied
work.

In [ ]:
# The inflation, computed directly.
print(f'{"tests":>7}{"P(at least one false alarm)":>32}')
print('-' * 40)
for k in [1, 2, 4, 10, 20, 50]:
    print(f'{k:>7}{1 - 0.95 ** k:>31.1%}')
print()
print('At 4 tests the family-wise error rate is already ~19%, not 5%.')

## Simulating it

The formula is easy to nod at and easy to disbelieve. This simulates four tests
on pure noise, ten thousand times, and counts how often at least one "finds"
something.

In [ ]:
rng = np.random.default_rng(42)
N_SIM, N_TESTS, N_OBS = 10000, 4, 30

false_alarms = 0
for _ in range(N_SIM):
    ps = [stats.ttest_1samp(rng.normal(0, 1, N_OBS), 0)[1] for _ in range(N_TESTS)]
    if min(ps) < 0.05:
        false_alarms += 1

print(f'{N_SIM:,} simulations of {N_TESTS} tests on data with NO real effect')
print(f'  at least one p < 0.05: {false_alarms / N_SIM:.1%}')
print(f'  theory says:           {1 - 0.95 ** N_TESTS:.1%}')

In [ ]:
csd = df.dropna(subset=['csd_code'])

imm = csd[csd['immigrant_status'] == 'Immigrant'][
    ['csd_code', 'geography_name', 'cma', 'Renter']].copy()
imm.columns = ['csd_code', 'geography_name', 'cma', 'renter_stir_imm']

nim = csd[csd['immigrant_status'] == 'Non-immigrants'][['csd_code', 'Renter']].copy()
nim.columns = ['csd_code', 'renter_stir_nim']

penalty_df = imm.merge(nim, on='csd_code', how='inner')
penalty_df['penalty'] = penalty_df['renter_stir_imm'] - penalty_df['renter_stir_nim']
penalty_df = penalty_df.dropna(subset=['penalty'])
penalty_df = penalty_df[penalty_df['cma'].isin(CITIES)]

print(f'{len(penalty_df)} CSDs where BOTH groups have a reported renter STIR')
print()
print(penalty_df[['geography_name', 'cma', 'renter_stir_imm',
                  'renter_stir_nim', 'penalty']].head(8).to_string(index=False))

## Applying the correction

Bonferroni divides the threshold by the number of tests. With four cities,
α = 0.05 becomes **0.0125**. It is the most conservative correction available and
the easiest to defend, because there is nothing to argue about in it.

In [ ]:
ALPHA = 0.05
N_CITY_TESTS = len(CITIES)
BONF = ALPHA / N_CITY_TESTS

print(f'uncorrected threshold: {ALPHA}')
print(f'Bonferroni threshold:  {ALPHA} / {N_CITY_TESTS} = {BONF}')
print()
print(f'{"City":<12}{"n":>5}{"mean gap":>11}{"p":>11}{"< 0.05":>9}{"< 0.0125":>11}')
print('-' * 59)
for city in CITIES:
    s = penalty_df[penalty_df['cma'] == city]['penalty']
    _, p = stats.ttest_1samp(s, 0)
    print(f'{city:<12}{len(s):>5}{s.mean():>+11.2f}{p:>11.5f}'
          f'{("yes" if p < ALPHA else "no"):>9}{("yes" if p < BONF else "no"):>11}')

## The result

**Edmonton survives, with a negative gap.** It is the only statistically robust
finding in this dataset, and it says immigrant renters in Edmonton spend a
*smaller* share of income on housing than non-immigrant renters in the same
municipalities.

That is the opposite of the "immigrant penalty" the earlier modules were
assembling. Rigour did not confirm the expected story; it contradicted it. Being
willing to report that is the skill this module is actually teaching.

### 🔧 Your turn 1

Change `ALPHA` to `0.10` and re-run.

Does any new city cross the uncorrected threshold? Does Edmonton's corrected
conclusion change? What does that tell you about how much the exact threshold
matters for strong versus borderline results?

In [ ]:
# Bonferroni is conservative. Holm is uniformly more powerful and just as valid.
ps = []
for city in CITIES:
    s = penalty_df[penalty_df['cma'] == city]['penalty']
    ps.append((city, stats.ttest_1samp(s, 0)[1]))

ordered = sorted(ps, key=lambda x: x[1])
print('Holm step-down — compare each p against alpha/(k - i)')
print()
print(f'{"rank":>5}  {"City":<13}{"p":>11}{"threshold":>12}{"reject?":>10}')
print('-' * 53)
k = len(ordered)
still_rejecting = True
for i, (city, p) in enumerate(ordered):
    thresh = ALPHA / (k - i)
    if still_rejecting and p < thresh:
        verdict = 'yes'
    else:
        still_rejecting = False
        verdict = 'no'
    print(f'{i + 1:>5}  {city:<13}{p:>11.5f}{thresh:>12.5f}{verdict:>10}')

### 🔧 Your turn 2

Holm and Bonferroni reach the same conclusion here.

Construct the case where they would differ: what would a second city's p-value
have to be for Holm to reject it while Bonferroni did not?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** At α = 0.10 no new city crosses: Montréal (0.36), Toronto (0.45)
and Vancouver (0.66) are far above even the looser threshold. Loosening α only
rescues borderline results — roughly those between 0.05 and 0.10 — and none of
these are borderline. Edmonton's corrected threshold becomes 0.10/4 = 0.025, and
its p of 0.0057 clears both. A strong result is insensitive to the correction
convention; that insensitivity is itself worth reporting.

**Your turn 2.** Bonferroni compares every p against 0.0125. Holm compares the
smallest against 0.05/4 = 0.0125, the second smallest against 0.05/3 = 0.0167,
and so on. So a second city with a p-value between 0.0125 and 0.0167 would be
rejected by Holm and not by Bonferroni. That gap is where Holm's extra power
lives, and it costs nothing in validity — which is why Holm is the better default
even though Bonferroni is easier to explain.

</details>

## Where this stops

You have one robust finding. You have no idea whether it is large enough to care
about — p-values are silent on magnitude. That is the next notebook.